In [1]:
import time
import math
import random
import numpy as np
from collections import defaultdict, Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

# ============================================================
# 1. Reproducibility
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ============================================================
# 2. Device
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ============================================================
# 3. Hyperparameters
# ============================================================
BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 1e-3

INPUT_SIZE = 28 * 28
NUM_CLASSES = 10

# MLP
HIDDEN1 = 256
HIDDEN2 = 128

# WiSARD
# Number of bits grouped into one RAM address
# Common values to try: 4, 8, 16
ADDRESS_SIZE = 8

# ============================================================
# 4. Dataset utilities
# ============================================================
def get_mnist_datasets():
    """
    Returns:
        train_gray, val_gray, test_gray
        train_bin,  val_bin,  test_bin
    """
    transform_gray = transforms.Compose([
        transforms.ToTensor(),  # [0,1]
        transforms.Normalize((0.1307,), (0.3081,))
    ])

    transform_bin = transforms.Compose([
        transforms.ToTensor(),
        transforms.Lambda(lambda x: (x > 0.5).float())
    ])

    full_train_gray = datasets.MNIST(
        root="./data",
        train=True,
        download=True,
        transform=transform_gray
    )
    test_gray = datasets.MNIST(
        root="./data",
        train=False,
        download=True,
        transform=transform_gray
    )

    full_train_bin = datasets.MNIST(
        root="./data",
        train=True,
        download=True,
        transform=transform_bin
    )
    test_bin = datasets.MNIST(
        root="./data",
        train=False,
        download=True,
        transform=transform_bin
    )

    train_size = int(0.9 * len(full_train_gray))
    val_size = len(full_train_gray) - train_size

    # same split sizes for both
    train_gray, val_gray = random_split(
        full_train_gray,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(SEED)
    )

    train_bin, val_bin = random_split(
        full_train_bin,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(SEED)
    )

    return train_gray, val_gray, test_gray, train_bin, val_bin, test_bin


def make_loaders(train_ds, val_ds, test_ds, batch_size=128):
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader


# ============================================================
# 5. MLP model
# ============================================================
class MLP(nn.Module):
    def __init__(self, input_size=784, hidden1=256, hidden2=128, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_size, hidden1),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden2, num_classes)
        )

    def forward(self, x):
        return self.net(x)


def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    return epoch_loss, epoch_acc


@torch.no_grad()
def evaluate_torch_model(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    start_inf = time.perf_counter()

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    end_inf = time.perf_counter()

    avg_loss = running_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="macro", zero_division=0
    )
    cm = confusion_matrix(all_labels, all_preds)

    total_inf_time = end_inf - start_inf
    avg_inf_time_per_sample = total_inf_time / len(loader.dataset)

    return {
        "loss": avg_loss,
        "accuracy": acc,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
        "confusion_matrix": cm,
        "y_true": np.array(all_labels),
        "y_pred": np.array(all_preds),
        "inference_time_total_sec": total_inf_time,
        "inference_time_per_sample_sec": avg_inf_time_per_sample,
    }


def run_mlp_experiment(train_loader, val_loader, test_loader, title="MLP"):
    model = MLP(INPUT_SIZE, HIDDEN1, HIDDEN2, NUM_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    best_val_acc = -1.0
    best_state = None

    train_start = time.perf_counter()

    for epoch in range(EPOCHS):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_results = evaluate_torch_model(model, val_loader, criterion, device)
        val_acc = val_results["accuracy"]

        print(f"{title} | Epoch {epoch+1}/{EPOCHS} | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Val Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    train_end = time.perf_counter()
    training_time = train_end - train_start

    model.load_state_dict(best_state)
    test_results = evaluate_torch_model(model, test_loader, criterion, device)
    test_results["training_time_sec"] = training_time
    test_results["num_parameters"] = count_trainable_parameters(model)
    test_results["model_name"] = title

    return model, test_results


# ============================================================
# 6. Simple WiSARD implementation
# ============================================================
class SimpleWiSARD:
    """
    A simple WiSARD-like classifier:
    - Input must be binary vector of shape [n_features]
    - Features are permuted once and split into RAM groups of address_size bits
    - For each class and RAM, observed addresses are stored
    - Prediction counts how many RAMs hit for each class

    This is a clean educational implementation for comparison experiments.
    """
    def __init__(self, input_size=784, num_classes=10, address_size=8, seed=42):
        self.input_size = input_size
        self.num_classes = num_classes
        self.address_size = address_size
        self.seed = seed

        self.rng = np.random.default_rng(seed)

        # Number of RAM nodes
        self.num_rams = math.ceil(input_size / address_size)

        # Pad input size if needed
        self.padded_size = self.num_rams * self.address_size
        self.padding = self.padded_size - self.input_size

        # One fixed random permutation for input mapping
        self.permutation = self.rng.permutation(self.padded_size)

        # discriminator[class_idx][ram_idx] = set(addresses seen)
        self.discriminators = [
            [set() for _ in range(self.num_rams)] for _ in range(self.num_classes)
        ]

    def _prepare_input(self, x):
        """
        x: 1D binary numpy array of shape [input_size]
        returns: permuted padded binary vector of shape [padded_size]
        """
        x = np.asarray(x).astype(np.uint8).reshape(-1)

        if len(x) != self.input_size:
            raise ValueError(f"Expected input size {self.input_size}, got {len(x)}")

        if self.padding > 0:
            x = np.concatenate([x, np.zeros(self.padding, dtype=np.uint8)])

        x = x[self.permutation]
        return x

    def _address_from_bits(self, bits):
        """
        Convert a small binary vector into integer address.
        Example: [1,0,1] -> 5
        """
        addr = 0
        for bit in bits:
            addr = (addr << 1) | int(bit)
        return addr

    def fit(self, X, y):
        """
        X: binary numpy array [N, input_size]
        y: labels [N]
        """
        for xi, yi in zip(X, y):
            xp = self._prepare_input(xi)

            for ram_idx in range(self.num_rams):
                start = ram_idx * self.address_size
                end = start + self.address_size
                bits = xp[start:end]
                addr = self._address_from_bits(bits)
                self.discriminators[int(yi)][ram_idx].add(addr)

    def score_sample(self, x):
        """
        Returns class scores = number of RAM hits per class
        """
        xp = self._prepare_input(x)
        scores = np.zeros(self.num_classes, dtype=np.int32)

        for ram_idx in range(self.num_rams):
            start = ram_idx * self.address_size
            end = start + self.address_size
            bits = xp[start:end]
            addr = self._address_from_bits(bits)

            for c in range(self.num_classes):
                if addr in self.discriminators[c][ram_idx]:
                    scores[c] += 1

        return scores

    def predict(self, X):
        preds = []
        for x in X:
            scores = self.score_sample(x)
            pred = int(np.argmax(scores))
            preds.append(pred)
        return np.array(preds)

    def memory_statistics(self):
        total_addresses = 0
        per_class_addresses = []

        for c in range(self.num_classes):
            class_total = sum(len(ram_set) for ram_set in self.discriminators[c])
            per_class_addresses.append(class_total)
            total_addresses += class_total

        return {
            "num_classes": self.num_classes,
            "num_rams": self.num_rams,
            "address_size": self.address_size,
            "total_stored_addresses": total_addresses,
            "stored_addresses_per_class": per_class_addresses,
        }


# ============================================================
# 7. Helpers to convert datasets to numpy for WiSARD
# ============================================================
def dataset_to_numpy_binary(dataset):
    """
    Converts a PyTorch dataset of tensors to:
      X: [N, 784] binary uint8
      y: [N]
    """
    X_list = []
    y_list = []

    for img, label in dataset:
        # img shape: [1, 28, 28]
        x = img.view(-1).numpy()
        x = (x > 0.5).astype(np.uint8)  # safety: enforce binary
        X_list.append(x)
        y_list.append(label)

    X = np.stack(X_list, axis=0)
    y = np.array(y_list)
    return X, y


def evaluate_numpy_classifier(model, X_test, y_test):
    start_inf = time.perf_counter()
    y_pred = model.predict(X_test)
    end_inf = time.perf_counter()

    acc = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="macro", zero_division=0
    )
    cm = confusion_matrix(y_test, y_pred)

    total_inf_time = end_inf - start_inf
    avg_inf_time_per_sample = total_inf_time / len(y_test)

    return {
        "accuracy": acc,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
        "confusion_matrix": cm,
        "y_true": y_test,
        "y_pred": y_pred,
        "inference_time_total_sec": total_inf_time,
        "inference_time_per_sample_sec": avg_inf_time_per_sample,
    }


def run_wisard_experiment(train_ds, test_ds, address_size=8, title="WiSARD"):
    X_train, y_train = dataset_to_numpy_binary(train_ds)
    X_test, y_test = dataset_to_numpy_binary(test_ds)

    model = SimpleWiSARD(
        input_size=INPUT_SIZE,
        num_classes=NUM_CLASSES,
        address_size=address_size,
        seed=SEED
    )

    train_start = time.perf_counter()
    model.fit(X_train, y_train)
    train_end = time.perf_counter()

    results = evaluate_numpy_classifier(model, X_test, y_test)
    results["training_time_sec"] = train_end - train_start
    results["memory_stats"] = model.memory_statistics()
    results["model_name"] = title

    return model, results


# ============================================================
# 8. Pretty printing
# ============================================================
def print_results(results):
    print("\n" + "=" * 70)
    print(f"Results for: {results['model_name']}")
    print("=" * 70)
    print(f"Accuracy                 : {results['accuracy']:.4f}")
    print(f"Macro Precision          : {results['precision_macro']:.4f}")
    print(f"Macro Recall             : {results['recall_macro']:.4f}")
    print(f"Macro F1                 : {results['f1_macro']:.4f}")
    print(f"Training Time (sec)      : {results['training_time_sec']:.4f}")
    print(f"Inference Time (sec)     : {results['inference_time_total_sec']:.4f}")
    print(f"Inference / Sample (sec) : {results['inference_time_per_sample_sec']:.8f}")

    if "num_parameters" in results:
        print(f"Trainable Parameters     : {results['num_parameters']}")

    if "memory_stats" in results:
        mem = results["memory_stats"]
        print(f"WiSARD RAM Count         : {mem['num_rams']}")
        print(f"WiSARD Address Size      : {mem['address_size']}")
        print(f"Stored Addresses Total   : {mem['total_stored_addresses']}")
        print(f"Stored Addresses/Class   : {mem['stored_addresses_per_class']}")

    print("\nConfusion Matrix:")
    print(results["confusion_matrix"])

    print("\nClassification Report:")
    print(classification_report(results["y_true"], results["y_pred"], digits=4))


def print_summary_table(results_list):
    print("\n" + "=" * 110)
    print("SUMMARY TABLE")
    print("=" * 110)
    header = (
        f"{'Model':<20}"
        f"{'Acc':>10}"
        f"{'Prec':>10}"
        f"{'Recall':>10}"
        f"{'F1':>10}"
        f"{'Train(s)':>12}"
        f"{'Infer(s)':>12}"
        f"{'Infer/sample':>16}"
    )
    print(header)
    print("-" * 110)

    for r in results_list:
        line = (
            f"{r['model_name']:<20}"
            f"{r['accuracy']:>10.4f}"
            f"{r['precision_macro']:>10.4f}"
            f"{r['recall_macro']:>10.4f}"
            f"{r['f1_macro']:>10.4f}"
            f"{r['training_time_sec']:>12.4f}"
            f"{r['inference_time_total_sec']:>12.4f}"
            f"{r['inference_time_per_sample_sec']:>16.8f}"
        )
        print(line)
    print("=" * 110)


# ============================================================
# 9. Main experiment
# ============================================================
def main():
    train_gray, val_gray, test_gray, train_bin, val_bin, test_bin = get_mnist_datasets()

    train_loader_gray, val_loader_gray, test_loader_gray = make_loaders(
        train_gray, val_gray, test_gray, batch_size=BATCH_SIZE
    )
    train_loader_bin, val_loader_bin, test_loader_bin = make_loaders(
        train_bin, val_bin, test_bin, batch_size=BATCH_SIZE
    )

    results_all = []

    # --------------------------------------------------------
    # A. Standard MLP on grayscale MNIST
    # --------------------------------------------------------
    mlp_gray_model, mlp_gray_results = run_mlp_experiment(
        train_loader_gray, val_loader_gray, test_loader_gray,
        title="MLP_Grayscale"
    )
    print_results(mlp_gray_results)
    results_all.append(mlp_gray_results)

    # --------------------------------------------------------
    # B. Fair comparison: MLP on binary MNIST
    # --------------------------------------------------------
    mlp_bin_model, mlp_bin_results = run_mlp_experiment(
        train_loader_bin, val_loader_bin, test_loader_bin,
        title="MLP_Binary"
    )
    print_results(mlp_bin_results)
    results_all.append(mlp_bin_results)

    # --------------------------------------------------------
    # C. WiSARD on binary MNIST
    # --------------------------------------------------------
    wisard_model, wisard_results = run_wisard_experiment(
        train_bin, test_bin,
        address_size=ADDRESS_SIZE,
        title=f"WiSARD_Binary_a{ADDRESS_SIZE}"
    )
    print_results(wisard_results)
    results_all.append(wisard_results)

    # --------------------------------------------------------
    # D. Final summary
    # --------------------------------------------------------
    print_summary_table(results_all)


if __name__ == "__main__":
    main()

Using device: cpu
MLP_Grayscale | Epoch 1/10 | Train Loss: 0.3363 | Train Acc: 0.8970 | Val Acc: 0.9567
MLP_Grayscale | Epoch 2/10 | Train Loss: 0.1413 | Train Acc: 0.9567 | Val Acc: 0.9678
MLP_Grayscale | Epoch 3/10 | Train Loss: 0.1062 | Train Acc: 0.9674 | Val Acc: 0.9707
MLP_Grayscale | Epoch 4/10 | Train Loss: 0.0826 | Train Acc: 0.9744 | Val Acc: 0.9743
MLP_Grayscale | Epoch 5/10 | Train Loss: 0.0715 | Train Acc: 0.9777 | Val Acc: 0.9765
MLP_Grayscale | Epoch 6/10 | Train Loss: 0.0624 | Train Acc: 0.9802 | Val Acc: 0.9747
MLP_Grayscale | Epoch 7/10 | Train Loss: 0.0551 | Train Acc: 0.9822 | Val Acc: 0.9757
MLP_Grayscale | Epoch 8/10 | Train Loss: 0.0514 | Train Acc: 0.9831 | Val Acc: 0.9785
MLP_Grayscale | Epoch 9/10 | Train Loss: 0.0456 | Train Acc: 0.9852 | Val Acc: 0.9773
MLP_Grayscale | Epoch 10/10 | Train Loss: 0.0426 | Train Acc: 0.9858 | Val Acc: 0.9765

Results for: MLP_Grayscale
Accuracy                 : 0.9806
Macro Precision          : 0.9804
Macro Recall             